# Lesson 10: Debugging and interpreting models

Goes with the Lesson 10 page of the Forkcast course. Confident mistakes, class balance, forward hooks, Grad-CAM, and calibration, first on the synthetic shapes, then on dish photos.

Runtime: a **GPU** for the dish photos; the shapes run anywhere.

## 1. Clone your repository and install the package

In [ ]:
GITHUB_USER = "your-github-username"   # edit this line
REPO = "forkcast-course"

!git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!pip install -e .

## 2. `forkcast/debug.py`

`reliability_bins` and `gallery` are provided. Write `misclassified`, `class_counts`, `class_weights`, `record_activation`, `GradCAM` and `expected_calibration_error` as the page builds them.

In [ ]:
%%writefile forkcast/debug.py
"""Lesson 10. Debugging and interpreting a classifier: look at the mistakes, look inside the model, check its confidence.

``reliability_bins`` and ``gallery`` are provided; the rest are exercises.
"""

from __future__ import annotations

import torch
from torch import Tensor, nn


def class_counts(labels: Tensor, n_classes: int) -> Tensor:
    """How many examples of each class: (N,) int64 -> (n_classes,) int64. torch.bincount with minlength."""
    raise NotImplementedError("Lesson 10, step 3: torch.bincount(labels, minlength=n_classes)")


def class_weights(counts: Tensor) -> Tensor:
    """Inverse-frequency weights: total / (n_classes * count), with empty classes clamped to 1.

    Every class then carries the same total weight (count × weight is equal for all classes), so a mistake
    on a rare class costs more than one on a common class when passed to nn.CrossEntropyLoss(weight=...).
    """
    raise NotImplementedError("Lesson 10, step 3: counts.sum() / (len(counts) * counts.clamp_min(1)), as float")


def misclassified(logits: Tensor, y: Tensor, k: int = 16) -> list[tuple[int, int, int, float]]:
    """The k most confidently wrong predictions as (index, true, predicted, confidence), highest confidence first.

    confidence is the softmax probability of the predicted class.
    """
    raise NotImplementedError("Lesson 10, step 2: softmax, max over classes, keep rows where predicted != y, sort by confidence descending, first k")


def record_activation(model: nn.Module, layer: nn.Module, x: Tensor) -> Tensor:
    """Run ``model(x)`` and return what ``layer`` produced on the way, using a forward hook.

    A forward hook is a function ``hook(module, inputs, output)`` registered with
    ``layer.register_forward_hook``; it is called every time the layer runs. Remove the hook
    afterwards with the handle it returns, or it will fire on every later forward pass forever.
    """
    raise NotImplementedError("Lesson 10, step 4: define hook, handle = layer.register_forward_hook(hook), model(x), handle.remove(), return captured output")


class GradCAM:
    """Where did the model look? A heatmap over ``layer``'s spatial grid for the chosen class.

    A forward hook captures the layer's activations (N, C, H, W). For the class score s, the
    gradient ds/dactivations is averaged over H and W to give one weight per channel; the
    heatmap is relu(sum over channels of weight * activation), normalised to [0, 1] per image.
    """

    def __init__(self, model: nn.Module, layer: nn.Module):
        self.model, self.layer, self.activations = model, layer, None
        raise NotImplementedError("Lesson 10, step 5: self.handle = layer.register_forward_hook(a method that stores output in self.activations)")

    def __call__(self, x: Tensor, class_idx: Tensor | None = None) -> Tensor:
        """(N, 3, H, W) -> (N, h, w) heatmaps in [0, 1]. ``class_idx`` defaults to the predicted class."""
        raise NotImplementedError("Lesson 10, step 5: x.clone().requires_grad_(True); forward under torch.enable_grad(); score; torch.autograd.grad; weights; relu; normalise")

    def remove(self) -> None:
        raise NotImplementedError("Lesson 10, step 5: self.handle.remove()")


def reliability_bins(probs: Tensor, y: Tensor, n_bins: int = 10) -> tuple[Tensor, Tensor, Tensor]:
    """Bin predictions by confidence. Returns (mean confidence, accuracy, count) per bin, each (n_bins,).

    ``probs`` is (N, C) after softmax. Empty bins have count 0 and confidence/accuracy 0.
    """
    confidence, predicted = probs.max(dim=1)
    correct = (predicted == y).float()
    edges = torch.linspace(0, 1, n_bins + 1)
    bin_idx = torch.bucketize(confidence, edges[1:-1], right=False).clamp(0, n_bins - 1)
    count = torch.bincount(bin_idx, minlength=n_bins).float()
    conf_sum = torch.bincount(bin_idx, weights=confidence, minlength=n_bins)
    acc_sum = torch.bincount(bin_idx, weights=correct, minlength=n_bins)
    safe = count.clamp_min(1)
    return conf_sum / safe, acc_sum / safe, count


def expected_calibration_error(probs: Tensor, y: Tensor, n_bins: int = 10) -> float:
    """Weighted mean, over bins, of |accuracy - confidence|, weights = fraction of predictions in the bin.

    0 means the model's confidence is exactly as often right as it claims. Use reliability_bins.
    """
    raise NotImplementedError("Lesson 10, step 6: conf, acc, count = reliability_bins(...); weighted sum of |acc - conf| with weights count / count.sum()")


def gallery(images: Tensor, picks: list[tuple[int, int, int, float]], nrow: int = 8) -> Tensor:
    """A make_grid tensor of the picked images (for plt.imshow(...permute(1, 2, 0)))."""
    from torchvision.utils import make_grid
    idx = torch.tensor([p[0] for p in picks], dtype=torch.long)
    return make_grid(images[idx].clamp(0, 1), nrow=nrow)

## 3. A trained Lesson 7 network, and its mistakes

In [ ]:
import importlib, torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from forkcast import vision, train, debug
from forkcast.vision import SHAPE_CLASSES

train_loader, val_loader = vision.shape_loaders(1000, 400)
torch.manual_seed(0)
model = vision.SmallCNN()
train.fit_v2(model, train_loader, val_loader, epochs=6, lr=1e-3, loss_fn=nn.CrossEntropyLoss())
images, y = vision.make_shapes(400, seed=1)
model.eval()
with torch.no_grad():
    logits = model(images)
print("accuracy", round((logits.argmax(1) == y).float().mean().item(), 3))

importlib.reload(debug)
picks = debug.misclassified(logits, y, k=16)
print([(i, SHAPE_CLASSES[t], SHAPE_CLASSES[p], round(c, 3)) for i, t, p, c in picks[:5]])
plt.figure(figsize=(10, 3)); plt.axis("off"); plt.imshow(debug.gallery(images, picks).permute(1, 2, 0))

## 4. Class balance

The shapes are balanced; build a lopsided set on purpose and see what weights do to per-class accuracy.

In [ ]:
importlib.reload(debug)
print("counts", debug.class_counts(y, 4), "weights", debug.class_weights(debug.class_counts(y, 4)))

x_all, y_all = vision.make_shapes(3000, seed=5)
keep = torch.cat([(y_all == 0).nonzero().flatten()[:400], (y_all == 1).nonzero().flatten()[:400], (y_all == 2).nonzero().flatten()[:400], (y_all == 3).nonzero().flatten()[:40]])
from torch.utils.data import DataLoader, TensorDataset
skew = DataLoader(TensorDataset(x_all[keep], y_all[keep]), batch_size=64, shuffle=True, drop_last=True)
counts = debug.class_counts(y_all[keep], 4); print("lopsided counts:", counts.tolist(), "weights:", debug.class_weights(counts))
for name, loss_fn in [("plain", nn.CrossEntropyLoss()), ("weighted", nn.CrossEntropyLoss(weight=debug.class_weights(counts)))]:
    torch.manual_seed(0); m = vision.SmallCNN()
    train.fit_v2(m, skew, val_loader, epochs=8, lr=1e-3, loss_fn=loss_fn)
    with torch.no_grad():
        p = m.eval()(images).argmax(1)
    print(f"{name:9s} per-class accuracy:", [round(((p == y) & (y == k)).sum().item() / (y == k).sum().item(), 2) for k in range(4)])

## 5. Hooks and Grad-CAM

In [ ]:
importlib.reload(debug)
for i in [0, 3, 6, 7]:
    print(f"features[{i}]:", tuple(debug.record_activation(model, model.features[i], images[:8]).shape))
print("hooks left on features[6]:", len(model.features[6]._forward_hooks))

cam = debug.GradCAM(model, model.features[6])
heat = cam(images[:8])
heat_wrong = cam(images[:8], class_idx=(y[:8] + 1) % 4)
cam.remove()
big = F.interpolate(heat[:, None], size=32, mode="bilinear", align_corners=False)[:, 0]
fig, axes = plt.subplots(2, 8, figsize=(14, 3.6))
for k in range(8):
    axes[0, k].imshow(images[k].permute(1, 2, 0)); axes[0, k].set_title(SHAPE_CLASSES[y[k]], fontsize=8); axes[0, k].axis("off")
    axes[1, k].imshow(images[k].permute(1, 2, 0)); axes[1, k].imshow(big[k], cmap="jet", alpha=0.5); axes[1, k].axis("off")
print("heatmap shape", tuple(heat.shape), "| same map for a different class?", torch.allclose(heat, heat_wrong))

## 6. Calibration

In [ ]:
importlib.reload(debug)
probs = torch.softmax(logits, dim=1)
print("ECE", round(debug.expected_calibration_error(probs, y), 3), "| mean confidence", round(probs.max(1).values.mean().item(), 3))
conf, acc, count = debug.reliability_bins(probs, y)
plt.figure(figsize=(4, 4)); plt.bar(conf[count > 0], acc[count > 0], width=0.08, alpha=0.7); plt.plot([0, 1], [0, 1], "k--"); plt.xlabel("confidence"); plt.ylabel("accuracy");

## 7. The same four tools on dish photos

Lesson 9's recipe, compressed: ten classes, a frozen pretrained ResNet-18, a head trained on cached features.

In [ ]:
!pip install -q datasets
from datasets import load_dataset
from torchvision import models
from forkcast import transfer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = models.ResNet18_Weights.DEFAULT; prep = weights.transforms()
KEEP = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]
stream = load_dataset("ethz/food101", split="train", streaming=True)
names = [stream.features["label"].names[k] for k in KEEP]
per_class = {k: 0 for k in KEEP}; xs, ys = [], []
for ex in stream:
    k = ex["label"]
    if k in per_class and per_class[k] < 120:
        xs.append(prep(ex["image"].convert("RGB"))); ys.append(KEEP.index(k)); per_class[k] += 1
    if all(v >= 120 for v in per_class.values()):
        break
X = torch.stack(xs); Y = torch.tensor(ys)
perm = torch.randperm(len(Y), generator=torch.Generator().manual_seed(0)); X, Y = X[perm], Y[perm]
food_train = DataLoader(TensorDataset(X[300:], Y[300:]), batch_size=64, shuffle=True); food_val = DataLoader(TensorDataset(X[:300], Y[:300]), batch_size=64)
dish = transfer.build_transfer_model(10).to(device)
feats, ys_tr = transfer.extract_features(dish, food_train, device); vfeats, ys_val = transfer.extract_features(dish, food_val, device)
head = nn.Linear(512, 10)
train.fit_v2(head, DataLoader(TensorDataset(feats, ys_tr), batch_size=128, shuffle=True), DataLoader(TensorDataset(vfeats, ys_val), batch_size=256), epochs=30, lr=1e-3, loss_fn=nn.CrossEntropyLoss(), device=device)
dish.fc.load_state_dict(head.state_dict()); dish.eval()
with torch.no_grad():
    dlogits = torch.cat([dish(x.to(device)).cpu() for x, _ in food_val])
dy = Y[:300]
print("accuracy", round((dlogits.argmax(1) == dy).float().mean().item(), 3), "| ECE", round(debug.expected_calibration_error(torch.softmax(dlogits, 1), dy), 3))
mean = torch.tensor(prep.mean)[:, None, None]; std = torch.tensor(prep.std)[:, None, None]
shown = (X[:300] * std + mean).clamp(0, 1)
picks = debug.misclassified(dlogits, dy, k=8)
print([(names[t], "->", names[p], round(c, 2)) for _, t, p, c in picks])
plt.figure(figsize=(12, 2)); plt.axis("off"); plt.title("most confident mistakes"); plt.imshow(debug.gallery(shown, picks).permute(1, 2, 0))

cam = debug.GradCAM(dish, dish.layer4)
sel = [p[0] for p in picks[:6]]
heat = cam(X[:300][sel].to(device)).cpu(); cam.remove()
big = F.interpolate(heat[:, None], size=224, mode="bilinear", align_corners=False)[:, 0]
fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
for k, i in enumerate(sel):
    axes[k].imshow(shown[i].permute(1, 2, 0)); axes[k].imshow(big[k], cmap="jet", alpha=0.45); axes[k].set_title(f"{names[dy[i]]} as {names[picks[k][2]]}", fontsize=8); axes[k].axis("off")

### Break it on purpose: hooks that stay

In [ ]:
stash = []
handles = [model.features[6].register_forward_hook(lambda m, i, o: stash.append(o)) for _ in range(50)]
with torch.no_grad():
    model(images[:8])
print("tensors kept alive by forgotten hooks:", len(stash), "| hooks on the layer:", len(model.features[6]._forward_hooks))
for h in handles:
    h.remove()
print("after removing:", len(model.features[6]._forward_hooks))

## 8. Definition of done

Expected: `7 passed`, then `76 passed` with no failures for everything so far (skipped tests belong to lessons you have not started).

In [ ]:
!pytest tests/test_10.py -q

In [ ]:
!pytest -m fast -q

## 9. Commit and push

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token with access to your repo: ")

!git config user.name "{GITHUB_USER}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
!git add -A
!git commit -m "Lesson 10: debugging and interpreting models"
!git push https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:main